# Feature Engineering — Previsão De Demanda

## Resumo Do Que Foi Feito

Neste notebook foi construída a base de features para o modelo de Previsão de Demanda com Random Forest Regressor.

A base principal utilizada foi a tabela Silver da Squad 2:

- `squad2/silver/ecommerce_pedidos`

Como complemento histórico, foram utilizadas duas bases Silver da Squad 3:

- `squad3/silver/ecommerce_pedidos`
- `squad3/silver/physical_vendas_caixa`

A Squad 2 foi mantida como fonte principal do projeto, enquanto a Squad 3 foi usada apenas para aumentar o volume histórico de dados e melhorar o aprendizado do modelo.

In [0]:
%run /Workspace/Users/kalitamariano01@gmail.com/merca-data-platform/notebooks/utils/utils_feat_squad2_99_helpers

## Tabelas Lidas

### Squad 2 — Base Principal

Tabela: `ecommerce_pedidos`

Colunas utilizadas:

- `id_pedido`
- `dt_pedido`
- `valor_total`
- `status_pedido`
- `metodo_pagamento`

Essas colunas foram usadas porque representam o evento de venda/pedido, o horário em que ocorreu, o valor da venda e informações básicas de contexto.

In [0]:
def get_squad2_client():
    return get_adls_client().get_file_system_client("squad2")


def ler_parquet_silver_squad2(nome_tabela: str):
    container_client = get_squad2_client()
    path_silver = f"silver/{nome_tabela}"

    paths = [
        p.name
        for p in container_client.get_paths(path=path_silver, recursive=True)
        if p.name.endswith(".parquet") and "_delta_log" not in p.name
    ]

    dfs = []

    for file_path in paths:
        file_client = container_client.get_file_client(file_path)
        data = file_client.download_file().readall()
        dfs.append(pd.read_parquet(io.BytesIO(data)))

    if not dfs:
        raise ValueError(f"Nenhum arquivo parquet encontrado em {path_silver}")

    df = pd.concat(dfs, ignore_index=True)

    for coluna in df.columns:
        if coluna.startswith("dt_") or coluna.endswith("_at"):
            df[coluna] = pd.to_datetime(df[coluna], errors="coerce")

    for coluna in df.columns:
        if df[coluna].apply(lambda x: isinstance(x, Decimal)).any():
            df[coluna] = df[coluna].astype(float)

    return spark.createDataFrame(df)

In [0]:
df_pedidos_squad2 = ler_parquet_silver_squad2("ecommerce_pedidos")

#display(df_pedidos_squad2.limit(10))

## Tabelas Complementares Da Squad 3

### Squad 3 — Ecommerce

Tabela: `ecommerce_pedidos`

Colunas utilizadas:

- `id_pedido`
- `dt_pedido`
- `valor_total`
- `status_pedido`
- `metodo_pagamento`

### Squad 3 — Loja Física

Tabela: `physical_vendas_caixa`

Colunas utilizadas:

- `id_transacao`
- `dt_venda`
- `valor_total_venda`
- `tipo_pagamento`

Essa base foi usada para complementar o histórico com vendas de loja física.

In [0]:
def get_squad3_client():
    return get_adls_client().get_file_system_client("squad3")


def ler_parquet_silver_squad3(nome_tabela: str):
    container_client = get_squad3_client()
    path_silver = f"silver/{nome_tabela}"

    paths = [
        p.name
        for p in container_client.get_paths(path=path_silver, recursive=True)
        if p.name.endswith(".parquet") and "_delta_log" not in p.name
    ]

    dfs = []

    for file_path in paths:
        file_client = container_client.get_file_client(file_path)
        data = file_client.download_file().readall()
        dfs.append(pd.read_parquet(io.BytesIO(data)))

    if not dfs:
        raise ValueError(f"Nenhum arquivo parquet encontrado em {path_silver}")

    df = pd.concat(dfs, ignore_index=True)

    for coluna in df.columns:
        if coluna.startswith("dt_") or coluna.endswith("_at"):
            df[coluna] = pd.to_datetime(df[coluna], errors="coerce")

    for coluna in df.columns:
        if df[coluna].apply(lambda x: isinstance(x, Decimal)).any():
            df[coluna] = df[coluna].astype(float)

    return spark.createDataFrame(df)

In [0]:
df_ecommerce_squad3 = ler_parquet_silver_squad3("ecommerce_pedidos")
df_physical_squad3 = ler_parquet_silver_squad3("physical_vendas_caixa")

# display(df_ecommerce_squad3.limit(10))
# display(df_physical_squad3.limit(10))

## Padronização Das Bases

Como as tabelas tinham nomes de colunas diferentes, todas foram transformadas para uma estrutura única:

- `id_evento`
- `dt_evento`
- `valor_venda`
- `status_venda`
- `metodo_pagamento`
- `origem_venda`

A coluna `origem_venda` foi criada para identificar de onde cada registro veio:

- `squad2_ecommerce`
- `squad3_ecommerce_batch`
- `squad3_physical_batch`

In [0]:
df_squad2_padrao = (
    df_pedidos_squad2
    .select(
        F.col("id_pedido").cast("string").alias("id_evento"),
        F.col("dt_pedido").alias("dt_evento"),
        F.col("valor_total").cast("double").alias("valor_venda"),
        F.col("status_pedido").alias("status_venda"),
        F.col("metodo_pagamento").alias("metodo_pagamento"),
        F.lit("squad2_ecommerce").alias("origem_venda")
    )
)

df_ecommerce_squad3_padrao = (
    df_ecommerce_squad3
    .select(
        F.col("id_pedido").cast("string").alias("id_evento"),
        F.col("dt_pedido").alias("dt_evento"),
        F.col("valor_total").cast("double").alias("valor_venda"),
        F.col("status_pedido").alias("status_venda"),
        F.col("metodo_pagamento").alias("metodo_pagamento"),
        F.lit("squad3_ecommerce_batch").alias("origem_venda")
    )
)

df_physical_squad3_padrao = (
    df_physical_squad3
    .select(
        F.col("id_transacao").cast("string").alias("id_evento"),
        F.col("dt_venda").alias("dt_evento"),
        F.col("valor_total_venda").cast("double").alias("valor_venda"),
        F.lit("Venda Loja Física").alias("status_venda"),
        F.col("tipo_pagamento").alias("metodo_pagamento"),
        F.lit("squad3_physical_batch").alias("origem_venda")
    )
)

In [0]:
df_vendas_unificado = (
    df_squad2_padrao
    .unionByName(df_ecommerce_squad3_padrao, allowMissingColumns=True)
    .unionByName(df_physical_squad3_padrao, allowMissingColumns=True)
)

#display(df_vendas_unificado.limit(20))

## Features Criadas

Após unir as bases, os dados foram agregados por data e hora para criar variáveis numéricas usadas pelo modelo.

Features criadas:

- `data_evento`: data da venda/pedido.
- `hora_evento`: hora em que a venda ocorreu.
- `dia_semana`: dia da semana da venda.
- `vendas_hora`: quantidade de vendas naquela hora.
- `receita_hora`: valor total vendido naquela hora.
- `ticket_medio_hora`: média de valor por venda naquela hora.
- `vendas_acumuladas`: total de vendas acumuladas no dia até aquela hora.
- `receita_acumulada`: receita acumulada no dia até aquela hora.
- `tempo_restante_dia`: quantidade de horas restantes até o fim do dia.
- `percentual_dia_decorrido`: percentual do dia já percorrido.

In [0]:
df_base = (
    df_vendas_unificado
    .withColumn("dt_evento", F.to_timestamp("dt_evento"))
    .withColumn("data_evento", F.to_date("dt_evento"))
    .withColumn("hora_evento", F.hour("dt_evento"))
    .withColumn("dia_semana", F.dayofweek("dt_evento"))
    .withColumn("valor_venda", F.col("valor_venda").cast("double"))
    .filter(F.col("dt_evento").isNotNull())
    .filter(F.col("valor_venda").isNotNull())
)

df_hora = (
    df_base
    .groupBy("data_evento", "hora_evento", "dia_semana")
    .agg(
        F.countDistinct("id_evento").alias("vendas_hora"),
        F.sum("valor_venda").alias("receita_hora"),
        F.avg("valor_venda").alias("ticket_medio_hora")
    )
)

janela_dia = (
    Window
    .partitionBy("data_evento")
    .orderBy("hora_evento")
    .rowsBetween(Window.unboundedPreceding, Window.currentRow)
)

janela_total_dia = Window.partitionBy("data_evento")

df_features_demanda = (
    df_hora
    .withColumn("vendas_acumuladas", F.sum("vendas_hora").over(janela_dia))
    .withColumn("receita_acumulada", F.sum("receita_hora").over(janela_dia))
    .withColumn("total_vendas_dia", F.sum("vendas_hora").over(janela_total_dia))
    .withColumn("tempo_restante_dia", F.lit(23) - F.col("hora_evento"))
    .withColumn("percentual_dia_decorrido", (F.col("hora_evento") + F.lit(1)) / F.lit(24))
)

#display(df_features_demanda)

## Tabela Criada

Ao final do notebook, foi criada a tabela:

- `squad2.ml_features_previsao_demanda_vendas_hora_v1`

Essa tabela será usada como entrada no próximo notebook, responsável pelo treinamento do modelo Random Forest Regressor.

In [0]:
database_ml = "squad2"
tabela_features = "ml_features_previsao_demanda_vendas_hora_v1"

spark.sql(f"CREATE DATABASE IF NOT EXISTS {database_ml}")

(
    df_features_demanda
    .write
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(f"{database_ml}.{tabela_features}")
)

#print(f"Tabela salva com sucesso: {database_ml}.{tabela_features}")

In [0]:
df_teste_features = spark.table("squad2.ml_features_previsao_demanda_vendas_hora_v1")

display(df_teste_features.limit(10))